# NOURA EL KHOLTI

## Random Forest

Random Forest est un algorithme d’apprentissage automatique basé sur un ensemble d’arbres de décision. Il améliore la précision et réduit le surapprentissage en combinant les prédictions de plusieurs arbres entraînés sur des sous-ensembles aléatoires des données.

### 1. Fondements Théoriques

Le cœur de l'algorithme repose sur la réduction de l'incertitude.
Mesure de l'EntropieL'entropie de Shannon quantifie le désordre dans les étiquettes de classe $S$:
$$H(S) = -\sum_{i=1}^{n} p_i \log_2(p_i)$$
Gain d'InformationLe gain d'information permet de mesurer l'efficacité d'un attribut $A$ pour classer les données:
$$Gain(S, A) = H(S) - \sum_{v \in Val(A)} \frac{|S_v|}{|S|} H(S_v)$$

### 2. Préparation de l'Environnement

In [ ]:
import pandas as pd
import numpy as np

def scinder_donnees(df, proportion_train=0.8):
    """Sépare manuellement les données en ensembles d'entraînement et de test."""
    df_melange = df.sample(frac=1, random_state=42).reset_index(drop=True)
    limite = int(len(df_melange) * proportion_train)
    
    train = df_melange.iloc[:limite]
    test = df_melange.iloc[limite:]
    return train, test

### 3. Fonctions de Calcul de l'Impureté

Ces fonctions permettent d'évaluer mathématiquement la qualité de chaque division potentielle de l'arbre.

In [ ]:
def calculer_entropie(cible):
    """Calcule l'entropie d'un vecteur de labels."""
    stats = cible.value_counts(normalize=True)
    return -np.sum(stats * np.log2(stats + 1e-9)) # +1e-9 pour éviter log(0)

def calculer_gain_info(df, colonne_test, colonne_cible):
    """Calcule le gain obtenu en divisant selon 'colonne_test'."""
    entropie_initiale = calculer_entropie(df[colonne_cible])
    
    valeurs, effectifs = np.unique(df[colonne_test], return_counts=True)
    entropie_ponderee = 0
    
    for i in range(len(valeurs)):
        sous_ensemble = df[df[colonne_test] == valeurs[i]]
        poids = effectifs[i] / len(df)
        entropie_ponderee += poids * calculer_entropie(sous_ensemble[colonne_cible])
        
    return entropie_initiale - entropie_ponderee

### 4. Algorithme de Construction Récursive

L'arbre est représenté sous forme de dictionnaire imbriqué. Le processus s'arrête lorsqu'un nœud est pur ou qu'il n'y a plus d'attributs disponibles.

In [ ]:
def construire_id3(donnees, attributs, cible, defaut=None):
    # 1. Si l'ensemble est vide, retourner la valeur par défaut
    if donnees.empty:
        return defaut
    
    # 2. Si toutes les cibles sont identiques (nœud pur)
    if len(np.unique(donnees[cible])) == 1:
        return donnees[cible].iloc[0]
    
    # 3. Si plus d'attributs, retourner la classe majoritaire
    majorite = donnees[cible].value_counts().idxmax()
    if not attributs:
        return majorite
    
    # 4. Sélection du meilleur attribut
    gains = {attr: calculer_gain_info(donnees, attr, cible) for attr in attributs}
    meilleur_attr = max(gains, key=gains.get)
    
    # 5. Création de la structure récursive
    arbre = {meilleur_attr: {}}
    nouveaux_attributs = [a for a in attributs if a != meilleur_attr]
    
    for val in np.unique(donnees[meilleur_attr]):
        sous_df = donnees[donnees[meilleur_attr] == val]
        sous_arbre = construire_id3(sous_df, nouveaux_attributs, cible, majorite)
        arbre[meilleur_attr][val] = sous_arbre
        
    return arbre

### 5. Prédiction et Validation

Pour évaluer le modèle, nous parcourons l'arbre pour chaque instance du jeu de test.

In [ ]:
def predire(instance, arbre):
    """Parcourt l'arbre pour classer une instance donnée."""
    if not isinstance(arbre, dict):
        return arbre
    
    attribut = next(iter(arbre))
    valeur_instance = instance[attribut]
    
    if valeur_instance in arbre[attribut]:
        return predire(instance, arbre[attribut][valeur_instance])
    else:
        # Gère les valeurs inconnues en retournant une valeur par défaut
        return None 

def evaluer_precision(df_test, arbre, cible):
    """Calcule le taux de réussite du modèle."""
    predictions = df_test.apply(lambda x: predire(x, arbre), axis=1)
    exactitude = (predictions == df_test[cible]).mean()
    return exactitude * 100

### 6. Visualisation de l'Arbre

Cette fonction permet d'afficher la structure de manière hiérarchique et indentée.

In [ ]:
def afficher_arbre(arbre, indentation=""):
    """Affiche l'arbre de décision de façon lisible."""
    if not isinstance(arbre, dict):
        print(f"{indentation}  -> Décision : {arbre}")
        return

    for attribut, branches in arbre.items():
        print(f"{indentation}[ {attribut} ]")
        for valeur, sous_arbre in branches.items():
            print(f"{indentation}  |-- Valeur: {valeur}")
            afficher_arbre(sous_arbre, indentation + "  |  ")

### 7. Exemple de Test

Nous créons un petit jeu de données simulant une décision de "Jouer au Tennis" en fonction de la météo.

In [ ]:
# 1. Création d'un dataset d'exemple
data = {
    'Meteo': ['Soleil', 'Soleil', 'Nuageux', 'Pluie', 'Pluie', 'Pluie', 'Nuageux', 'Soleil'],
    'Temperature': ['Chaud', 'Chaud', 'Chaud', 'Frais', 'Froid', 'Froid', 'Froid', 'Frais'],
    'Humidite': ['Haute', 'Haute', 'Haute', 'Haute', 'Normale', 'Normale', 'Normale', 'Haute'],
    'Jouer': ['Non', 'Non', 'Oui', 'Oui', 'Oui', 'Non', 'Oui', 'Non']
}

df_exemple = pd.DataFrame(data)
attributs_list = ['Meteo', 'Temperature', 'Humidite']

# 2. Entraînement
mon_arbre = construire_id3(df_exemple, attributs_list, 'Jouer')

# 3. Affichage du résultat
print("Structure de l'arbre généré :")
afficher_arbre(mon_arbre)

```
Structure de l'arbre généré :
[ Meteo ]
  |-- Valeur: Nuageux
  |    -> Décision : Oui
  |-- Valeur: Pluie
  |  [ Temperature ]
  |    |-- Valeur: Frais
  |    |    -> Décision : Oui
  |    |-- Valeur: Froid
  |    |  [ Humidite ]
  |    |    |-- Valeur: Normale
  |    |    |    -> Décision : Oui
  |-- Valeur: Soleil
  |    -> Décision : Non
```

L'algorithme segmente efficacement les données pour aboutir à des décisions logiques et précises.